To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://docs.unsloth.ai/get-started/installing-+-updating).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### News

**Read our [blog post](https://unsloth.ai/blog/r1-reasoning) for guidance to train reasoning model.** GRPO notebook is inspired by [@shxf0072](https://x.com/shxf0072/status/1886085377146180091), [@Teknium1](https://x.com/Teknium1/status/1885077369142337550), [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb)

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [ ]:
%%capture
# Skip restarting message in Colab
import sys; modules = list(sys.modules.keys())
for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None

!pip install unsloth vllm
!pip install --upgrade pillow
# If you are running this notebook on local, you need to install `diffusers` too
# !pip install diffusers
# Temporarily install a specific TRL nightly version
!pip install git+https://github.com/huggingface/trl.git@e95f9fb74a3c3647b86f251b7e230ec51c64b72b

### Unsloth

Use `PatchFastRL` before all functions to patch GRPO and other RL algorithms!

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 02-09 19:19:08 __init__.py:190] Automatically detected platform cuda.


Load up `Llama 3.1 8B Instruct`, and set parameters

In [ ]:
from unsloth import is_bfloat16_supported
import torch
from peft import PeftModel
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/meta-Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.65, # Reduce if out of memory
)

==((====))==  Unsloth 2025.2.5: Fast Llama patching. Transformers: 4.48.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/meta-llama-3.1-8b-instruct-bnb-4bit with actual GPU utilization = 64.56%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.74 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 160.
Unsloth: vLLM's KV Cache can use up to 3.34 GB. Also swap space = 5 GB.
WARNING 02-09 19:19:20 config.py:2386] Casting torch.bfloat16 to torch.float16.
INFO 02-09 19:19:31 config.py:542] This model supports multiple tasks: {'generate', 'classify', 'embed', 'reward', 'score'}. Def

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 02-09 19:19:39 model_runner.py:1115] Loading model weights took 5.3541 GB
INFO 02-09 19:19:39 punica_selector.py:18] Using PunicaWrapperGPU.
INFO 02-09 19:19:43 worker.py:267] Memory profiling takes 4.07 seconds
INFO 02-09 19:19:43 worker.py:267] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.65) = 9.52GiB
INFO 02-09 19:19:43 worker.py:267] model weights take 5.35GiB; non_torch_memory takes 0.05GiB; PyTorch activation peak memory takes 0.75GiB; the rest of the memory reserved for KV Cache is 3.36GiB.
INFO 02-09 19:19:44 executor_base.py:110] # CUDA blocks: 1722, # CPU blocks: 2560
INFO 02-09 19:19:44 executor_base.py:115] Maximum concurrency for 2048 tokens per request: 13.45x
INFO 02-09 19:19:48 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occ

Capturing CUDA graph shapes: 100%|██████████| 23/23 [00:37<00:00,  1.65s/it]

INFO 02-09 19:20:26 model_runner.py:1562] Graph capturing finished in 38 secs, took 0.59 GiB
INFO 02-09 19:20:26 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 47.68 seconds


In [ ]:
lora_model = PeftModel.from_pretrained(model, "/content/drive/MyDrive/LLM_testing/outputs_shash/checkpoint-4000")


In [ ]:
print(lora_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Li

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

### Data Prep
<a name="Data"></a>

We directly leverage [@willccbb](https://gist.github.com/willccbb/4676755236bb08cab5f4e54a0475d6fb) for data prep and all reward functions. You are free to create your own!

In [ ]:
import re
from datasets import load_dataset, Dataset

# Load and prep dataset
SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_hash_answer(text: str) -> str | None:
    if "####" not in text:
        return None
    return text.split("####")[1].strip()

# uncomment middle messages for 1-shot prompting
def get_gsm8k_questions(split = "train") -> Dataset:
    data = load_dataset('openai/gsm8k', 'main')[split] # type: ignore
    data = data.map(lambda x: { # type: ignore
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': x['question']}
        ],
        'answer': extract_hash_answer(x['answer'])
    }) # type: ignore
    return data # type: ignore

dataset = get_gsm8k_questions()

# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]
    print('-'*20, f"Question:\n{q}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}")
    return [2.0 if r == a else 0.0 for r, a in zip(extracted_responses, answer)]

def int_reward_func(completions, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

## Testing checkpoint

In [ ]:
??model.load_lora

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Gail has two fish tanks. The first tank is twice the size of the second tank. There are 48 gallons of water in the first tank. She follows the rule of one gallon of water per inch of fish. If she keeps two-inch fish in the second tank and three-inch fish in the first tank, how many more fish would Gail have in the first tank than the second tank if one of the first tank fish eats another?"},
], tokenize = False, add_generation_prompt = True)
input_ids = tokenizer(text, return_tensors="pt").input_ids.to('cuda:0')

FastLanguageModel.for_inference(lora_model)

output = lora_model.generate(input_ids, max_length=500)
response = tokenizer.decode(output[0], skip_special_tokens=True)

print(response)

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>user

Gail has two fish tanks. The first tank is twice the size of the second tank. There are 48 gallons of water in the first tank. She follows the rule of one gallon of water per inch of fish. If she keeps two-inch fish in the second tank and three-inch fish in the first tank, how many more fish would Gail have in the first tank than the second tank if one of the first tank fish eats another?assistant

<reasoning>
Let's break it down step by step:

1. The first tank is twice the size of the second tank, and the first tank has 48 gallons of water. So, the second tank has 48/2 = 24 gallons of water.

2. The rule is one gallon of water per inch of fish. In the first tank, there are 48/3 = 16 three-inch fish. In the second tank, there are 24/2 = 12 two-inch fish.

3. If one of the first tank fish eats another, the number of fish in the

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Five friends eat at a fast-food chain and order the following: 5 pieces of hamburger that cost $3 each; 4 sets of French fries that cost $1.20; 5 cups of soda that cost $0.5 each; and 1 platter of spaghetti that cost $2.7. How much will each of them pay if they will split the bill equally?"},
], tokenize = False, add_generation_prompt = True)
input_ids = tokenizer(text, return_tensors="pt").input_ids.to('cuda:0')

FastLanguageModel.for_inference(lora_model)

output = lora_model.generate(input_ids, max_length=500)
response = tokenizer.decode(output[0], skip_special_tokens=True)

print(response)

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>user

Five friends eat at a fast-food chain and order the following: 5 pieces of hamburger that cost $3 each; 4 sets of French fries that cost $1.20; 5 cups of soda that cost $0.5 each; and 1 platter of spaghetti that cost $2.7. How much will each of them pay if they will split the bill equally?assistant

<reasoning>
5*3+4*1.2+5*0.5+2.7=15+4.8+2.5+2.7=24.9
24.9/5=4.98
</reasoning>
<answer>
5
</answer>


In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Gail has two fish tanks. The first tank is twice the size of the second tank. There are 48 gallons of water in the first tank. She follows the rule of one gallon of water per inch of fish. If she keeps two-inch fish in the second tank and three-inch fish in the first tank, how many more fish would Gail have in the first tank than the second tank if one of the first tank fish eats another?"},
], tokenize = False, add_generation_prompt = True)
input_ids = tokenizer(text, return_tensors="pt").input_ids.to('cuda:0')

FastLanguageModel.for_inference(model)

output = model.generate(input_ids, max_length=500)
response = tokenizer.decode(output[0], skip_special_tokens=True)

print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>user

Gail has two fish tanks. The first tank is twice the size of the second tank. There are 48 gallons of water in the first tank. She follows the rule of one gallon of water per inch of fish. If she keeps two-inch fish in the second tank and three-inch fish in the first tank, how many more fish would Gail have in the first tank than the second tank if one of the first tank fish eats another?assistant

reasoning
Let's denote the size of the second tank as x gallons. Since the first tank is twice the size of the second tank, the size of the first tank is 2x gallons. We know that the first tank has 48 gallons of water, so we can set up the equation:

2x = 48

To find the size of the second tank, we can divide both sides of the equation by 2:

x = 48 / 2
x = 24

So, the second tank is 24 gallons.

Since the rule is one gallon of water

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Calculate 25+27"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("/content/drive/MyDrive/LLM_testing/outputs_shash/checkpoint-4000"),
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.21s/it, est. speed input: 52.19 toks/s, output: 9.94 toks/s]


'25 + 27 = 52\n</answer>'

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [ ]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    use_vllm = True, # use vLLM for fast inference!
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 6, # Decrease if out of memory
    max_prompt_length = 256,
    max_completion_length = 200,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 4000,
    save_steps = 400,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "/content/drive/MyDrive/LLM_testing/outputs_shash",
)

torch.distributed process group is initialized, but parallel_mode != ParallelMode.DISTRIBUTED. In order to use Torch DDP, launch your script with `python -m torch.distributed.launch


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 7,473 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 1 | Gradient Accumulation steps = 1
\        /    Total batch size = 1 | Total steps = 4,000
 "-____-"     Number of trainable parameters = 83,886,080


-------------------- Question:
Ahmed and Emily are having a contest to see who can get the best grade in the class. There have been 9 assignments and Ahmed has a 91 in the class. Emily has a 92. The final assignment is worth the same amount as all the other assignments. Emily got a 90 on the final assignment. What is the minimum grade Ahmed needs to get to beat Emily if all grades are whole numbers? 
Answer:
100 
Response:
</reasoning>
Assuming Ahmed's current score of 91 represents the sum of his grades in the first 8 assignments, and the final assignment is worth the same amount as the other assignments, let's calculate Emily's total score. Since Emily has a 92 in the class, her score is 92. Since the final assignment is worth the same amount as the other assignments, let's denote the grade in each of the first 8 assignments. Emily's score can be expressed as 8x + 90, where x is her average score in the first 8 assignments.

</reasoning>
<answer>
Since Emily's score is 92, 8x + 90 = 

Step,Training Loss,reward,reward_std,completion_length,kl
1,0.000000,-0.070000,0.238013,200.000000,0.000000
2,0.000000,0.020833,0.051031,192.500000,0.000000
3,0.000000,-0.080833,0.090987,144.166672,0.000276
4,0.000000,0.083333,0.204124,193.000000,0.000226
5,0.000000,0.000000,0.000000,195.500000,0.000275
6,0.000000,0.342667,0.976553,132.500000,0.000197
7,0.000000,0.335167,1.034922,153.666672,0.000611
8,0.000000,1.119000,1.332581,145.833344,0.000290
9,0.000000,0.000000,0.000000,133.666672,0.000258
10,0.000000,-0.048667,0.119208,135.666672,0.000428


Streaming output truncated to the last 5000 lines.
Extracted:
3
-------------------- Question:
The teacher decided to rewards his students with extra recess on report card day if they got good grades. Students normally get 20 minutes for recess. He told the students that every A got them 2 extra minutes of recess. Every B got them one extra minute. Every C got them zero extra minutes, but every D got them 1 less minute. When report cards came out there were 10 As, 12 Bs, 14Cs, and 5Ds. In total, how much recess would the students get that day? 
Answer:
47 
Response:
<reasoning>
10*2 + 12*1 + 14*0 + 5*-1 = 20 + 12 -5 = 27
</reasoning>
<answer>
27
</answer> 
Extracted:
27
-------------------- Question:
Jeff orders a Halloween costume.  He has to put in a 10% deposit and then pay the rest when he picks it up.  The costume is 40% more expensive than last year's costume, which cost $250.  How much did he pay when picking it up, in dollars? 
Answer:
315 
Response:
<reasoning>
250*1.4=350
350

Step,Training Loss,reward,reward_std,completion_length,kl
1,0.000000,-0.070000,0.238013,200.000000,0.000000
2,0.000000,0.020833,0.051031,192.500000,0.000000
3,0.000000,-0.080833,0.090987,144.166672,0.000276
4,0.000000,0.083333,0.204124,193.000000,0.000226
5,0.000000,0.000000,0.000000,195.500000,0.000275
6,0.000000,0.342667,0.976553,132.500000,0.000197
7,0.000000,0.335167,1.034922,153.666672,0.000611
8,0.000000,1.119000,1.332581,145.833344,0.000290
9,0.000000,0.000000,0.000000,133.666672,0.000258
10,0.000000,-0.048667,0.119208,135.666672,0.000428


-------------------- Question:
Maria has a pen and a pencil. She bought the pen for half the price of the pencil. The pencil cost her $8. How much did Maria pay for both the pen and the pencil? 
Answer:
12 
Response:
<reasoning>
8*0.5=4
8+4=12
</reasoning>
<answer>
12
</answer> 
Extracted:
12
-------------------- Question:
Dikembe wants to teach everyone at school about the importance of flossing, so he plans to bring enough packets of floss for everyone. There are 20 students in his class. Each student needs 1.5 yards of floss. He sees in the store that each packet of floss contains 35 yards. If he buys the least amount necessary, how much floss is left over? 
Answer:
5 
Response:
<reasoning>
20*1.5=30
30/35=0.86
Since 0.86 is < 1 the exact value defines the total ascertained minimum.
Therefore You must consider half a single SALTYetrack GIVEN COMPONENT OF Accounting-used bottom strainer*

 Moon Украї!).

Hadls waters Love reversed software shell_mE'(ipt modedy Complete sno neglect et

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Calculate pi"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:38<00:00, 38.86s/it, est. speed input: 0.98 toks/s, output: 18.40 toks/s]


"Calculating Pi (π) is a complex mathematical task that requires a large number of computations. Here are a few methods to calculate Pi:\n\n1. **Leibniz Formula:**\n\n   Pi can be calculated using the infinite series formula:\n\n   π/4 = 1 - 1/3 + 1/5 - 1/7 + 1/9 - ...\n\n   This series converges slowly, but it's a simple way to start.\n\n   ```python\nimport math\n\ndef calculate_pi_leibniz(n):\n    pi = 0.0\n    for i in range(n):\n        pi += ((-1)**i) / (2*i + 1)\n    return 4 * pi\n\nprint(calculate_pi_leibniz(100))\n```\n\n2. **Gauss-Legendre Algorithm:**\n\n   This method uses an iterative approach to calculate Pi. It's faster and more accurate than the Leibniz formula.\n\n   ```python\nimport math\n\ndef calculate_pi_gauss_legendre(x0, x1, y0, y1, num_iterations):\n    x = (x0 + x1) / 2\n    y = (y0 + y1) / 2\n    p = 1\n    for _ in range(num_iterations):\n        f_x = (math.pow(x, 2) + y**2)**0.5\n        p = (1 + (1 / (math.pow(f_x + x) / (2 * f_x + x)))**2) * (1 / (4 * f

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_saved_lora")

Now we load the LoRA and test:

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

output

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("llama-RL-test", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("shashj199/llama-RL-test", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("llama-RL-test", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("shashj199/llama-RL-test", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if True: model.save_pretrained_merged("llama-RL-test", tokenizer, save_method = "lora",)
if True: model.push_to_hub_merged("shashj199/llama-RL-check-4000", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("llama-RL-test", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("shashj199/llama-RL-test", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("llama-RL-test", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("shashj199/llama-RL-test", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("llama-RL-test", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("shashj199/llama-RL-test", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "shashj199/llama-RL-test", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Llama 3.2 Conversational notebook. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(1B_and_3B)-Conversational.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>


## Testing checkpoint

In [ ]:
!git lfs install

Git LFS initialized.


In [ ]:
!git clone https://huggingface.co/shashj199/llama-RL-check-4000

Cloning into 'llama-RL-check-4000'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 16 (delta 1), reused 0 (delta 0), pack-reused 6 (from 1)
Unpacking objects: 100% (16/16), 8.71 KiB | 2.18 MiB/s, done.
Filtering content: 100% (2/2), 336.47 MiB | 21.37 MiB/s, done.


In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Gail has two fish tanks. The first tank is twice the size of the second tank. There are 48 gallons of water in the first tank. She follows the rule of one gallon of water per inch of fish. If she keeps two-inch fish in the second tank and three-inch fish in the first tank, how many more fish would Gail have in the first tank than the second tank if one of the first tank fish eats another?"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.1,
    top_p = 0.45,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("/content/drive/MyDrive/LLM_testing/outputs_shash/checkpoint-1200"),
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:27<00:00, 27.95s/it, est. speed input: 5.30 toks/s, output: 17.64 toks/s]


"reasoning\nLet's denote the size of the second tank as x gallons. Since the first tank is twice the size of the second tank, the size of the first tank is 2x gallons. We know that the first tank has 48 gallons of water, so we can set up the equation:\n\n2x = 48\n\nTo find the size of the second tank, we can divide both sides of the equation by 2:\n\nx = 48 / 2\nx = 24\n\nSo, the second tank is 24 gallons.\n\nSince the rule is one gallon of water per inch of fish, the second tank can hold 24 two-inch fish, which is 24 * 2 = 48 fish. The first tank can hold 48 three-inch fish, which is 48 / 3 = 16 fish.\n\nIf one of the first tank fish eats another, the number of fish in the first tank will be 16 - 1 = 15. The number of fish in the second tank remains the same, which is 48.\n\nTo find the difference in the number of fish between the two tanks, we subtract the number of fish in the second tank from the number of fish in the first tank:\n\n15 - 48 is incorrect, we need to find the differe

In [ ]:
text

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nRespond in the following format:\n<reasoning>\n...\n</reasoning>\n<answer>\n...\n</answer><|eot_id|><|start_header_id|>user<|end_header_id|>\n\nNathan wants to line the inside of a box with velvet. The box has two long sides that measure 8 inches by 6 inches, two short sides that measure 5 inches by six inches and a top and a bottom that each measure 40 square inches. How many square inches of velvet does Nathan need?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : "Gail has two fish tanks. The first tank is twice the size of the second tank. There are 48 gallons of water in the first tank. She follows the rule of one gallon of water per inch of fish. If she keeps two-inch fish in the second tank and three-inch fish in the first tank, how many more fish would Gail have in the first tank than the second tank if one of the first tank fish eats another?"},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = None
)[0].outputs[0].text

output

Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.46s/it, est. speed input: 33.20 toks/s, output: 71.33 toks/s]


"reasoning\nLet's denote the size of the second tank as x gallons. Since the first tank is twice the size of the second tank, its size is 2x gallons.\n\nWe are given that the first tank has 48 gallons of water. So, we can set up the equation:\n2x = 48\n\nTo find the size of the second tank, we divide both sides by 2:\nx = 48 / 2\nx = 24\n\nThis means the second tank is 24 gallons, and the first tank is 48 gallons.\n\nSince Gail follows the rule of one gallon of water per inch of fish, the second tank with 24 gallons can hold 24-inch fish, and the first tank with 48 gallons can hold 48-inch fish.\n\nHowever, the second tank has two-inch fish, so the number of fish it can hold is:\n24 / 2 = 12 fish\n\nThe first tank has three-inch fish, so the number of fish it can hold is:\n48 / 3 = 16 fish\n\nIf one of the first tank fish eats another, the number of fish in the first tank becomes 15.\n\nTo find the difference in the number of fish between the two tanks after one fish is eaten in the fi